**Lab 6: Giới thiệu về Transformers**

In [ ]:
! pip install transformers torch tf-keras hf_xet

## Bài 1: Khôi phục Masked Token (Masked Language Modeling)
Câu hỏi:
1. Mô hình đã dự đoán đúng từ capital không? - Có, mô hình dự đoán từ 'capital' với độ tin cậy cao nhất: 0.9952
2. Tại sao các mô hình Encoder-only như BERT lại phù hợp cho tác vụ này? - Các mô hình Encoder-only (như BERT) rất phù hợp vì chúng được thiết kế để đọc thông tin từ cả hai phía (trước và sau) của token [MASK] nhờ cơ chế Self-Attention.Ngoài ra, tác vụ chính mà của các mô hình encoder-only được đào tạo là Masked Language Modeling (MLM), tức là dự đoán các token bị che dựa trên ngữ cảnh còn lại.

In [8]:
from transformers import pipeline
# 1. Tải pipeline "fill-mask"
# Pipeline này sẽ tự động tải một mô hình mặc định phù hợp (thường là một biến thể của BERT)
mask_filler = pipeline("fill-mask", model="distilbert-base-uncased")
# # 2. Câu đầu vào với token [MASK]
input_sentence = "Hanoi is the [MASK] of Vietnam."
# 3. Thực hiện dự đoán
# top_k=5 yêu cầu mô hình trả về 5 dự đoán hàng đầu
predictions = mask_filler(input_sentence, top_k=5)
# 4. In kết quả
print(f"Câu gốc: {input_sentence}")
for pred in predictions:
    print(f"Dự đoán: '{pred['token_str']}' với độ tin cậy: {pred['score']:.4f}")
    print(f" -> Câu hoàn chỉnh: {pred['sequence']}")

Device set to use cpu


Câu gốc: Hanoi is the [MASK] of Vietnam.
Dự đoán: 'capital' với độ tin cậy: 0.9952
 -> Câu hoàn chỉnh: hanoi is the capital of vietnam.
Dự đoán: 'birthplace' với độ tin cậy: 0.0006
 -> Câu hoàn chỉnh: hanoi is the birthplace of vietnam.
Dự đoán: 'province' với độ tin cậy: 0.0005
 -> Câu hoàn chỉnh: hanoi is the province of vietnam.
Dự đoán: 'northernmost' với độ tin cậy: 0.0004
 -> Câu hoàn chỉnh: hanoi is the northernmost of vietnam.
Dự đoán: 'southernmost' với độ tin cậy: 0.0003
 -> Câu hoàn chỉnh: hanoi is the southernmost of vietnam.


## Bài 2: Dự đoán từ tiếp theo (Next Token Prediction)
Câu hỏi:
1. Kết quả sinh ra có hợp lý không? - Kết quả sinh ra có ngữ nghĩa và mạch lạc.
2. Tại sao các mô hình Decoder-only như GPT lại phù hợp cho tác vụ này? - Các mô hình Decoder-only (như GPT) phù hợp vì chúng được thiết kế chỉ sử dụng ngữ cảnh là các token đứng trước để dự đoán token tiếp theo, mô phỏng quá trình sinh văn bản tự nhiên. 

In [10]:
# 1. Tải pipeline "text-generation"
# Pipeline này sẽ tự động tải một mô hình phù hợp (thường là GPT-2)
generator = pipeline("text-generation", model="gpt2")
# 2. Đoạn văn bản mồi
prompt = "The best thing about learning NLP is"
# 3. Sinh văn bản
# max_length: tổng độ dài của câu mồi và phần được sinh ra
# num_return_sequences: số lượng chuỗi kết quả muốn nhận
generated_texts = generator(prompt, max_length=50, num_return_sequences=1)
# 4. In kết quả
print(f"Câu mồi: '{prompt}'")
for text in generated_texts:
    print("Văn bản được sinh ra:")
    print(text['generated_text'])

C:\Users\Admin\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Admin\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but th

Câu mồi: 'The best thing about learning NLP is'
Văn bản được sinh ra:
The best thing about learning NLP is that it's a lot easier to write about yourself. It's like learning how to read a book in the morning.

I'm happy that my NLP is a way to go about learning to be a better person, because it really helps me to be more positive, which is really important to me.

What I really love about NLP is that it takes time to grow. It's not a big deal. I can't say that I've ever been a great person, but it's something I really enjoy doing.

I'm very happy that my NLP is a way to go about growing. It's like learning how to read a book in the morning. I think that's very important to me.

I think it really helps me to be more positive and I think it's great to be able to learn new things in life. I think that's very important for me.

I think it's great to be able to be more positive and I think it's great to be able to be able to become more positive.

There are so many different ways I am going

# Bài 3: Tính toán Vector biểu diễn của câu (Sentence Representation)
Câu hỏi:
1. Kích thước (chiều) của vector biểu diễn là bao nhiêu? Con số này tương ứng với
tham số nào của mô hình BERT? - Chiều của vector biểu diễn là 768, tương ứng với hidden_size của mô hình BERT cơ bản.
2. Tại sao chúng ta cần sử dụng attention_mask khi thực hiện Mean Pooling? - Để loại bỏ các token đệm khỏi phép tính trung bình. Khi xử lý hàng loạt câu có độ dài khác nhau, các câu ngắn hơn được thêm token đệm ([PAD]) để đạt cùng độ dài với câu dài nhất. attention_mask chỉ ra những vị trí nào là token thực tế (giá trị 1) và những vị trí nào là token đệm (giá trị 0). Khi tính Mean Pooling (trung bình cộng), ta lấy trung bình của các vector token thực tế để đảm bảo vector biểu diễn câu không bị làm sai lệch bởi các giá trị 0 (giá trị đệm).

In [11]:
import torch
from transformers import AutoTokenizer, AutoModel

# 1. Chọn một mô hình BERT
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
# 2. Câu đầu vào
sentences = ["This is a sample sentence."]
# 3. Tokenize câu
# padding=True: đệm các câu ngắn hơn để có cùng độ dài
# truncation=True: cắt các câu dài hơn
# return_tensors='pt': trả về kết quả dưới dạng PyTorch tensors
inputs = tokenizer(sentences, padding=True, truncation=True, return_tensors='pt')
# 4. Đưa qua mô hình để lấy hidden states
# torch.no_grad() để không tính toán gradient, tiết kiệm bộ nhớ
with torch.no_grad():
    outputs = model(**inputs)
# outputs.last_hidden_state chứa vector đầu ra của tất cả các token
last_hidden_state = outputs.last_hidden_state
# shape: (batch_size, sequence_length, hidden_size)
# 5. Thực hiện Mean Pooling
# Để tính trung bình chính xác, chúng ta cần bỏ qua các token đệm (padding tokens)
attention_mask = inputs['attention_mask']
mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
sum_embeddings = torch.sum(last_hidden_state * mask_expanded, 1)
sum_mask = torch.clamp(mask_expanded.sum(1), min=1e-9)
sentence_embedding = sum_embeddings / sum_mask
# 6. In kết quả
print("Vector biểu diễn của câu:")
print(sentence_embedding)
print("\nKích thước của vector:", sentence_embedding.shape)

C:\Users\Admin\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Admin\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this

Vector biểu diễn của câu:
tensor([[-6.3874e-02, -4.2837e-01, -6.6779e-02, -3.8430e-01, -6.5785e-02,
         -2.1826e-01,  4.7636e-01,  4.8659e-01,  4.0311e-05, -7.4273e-02,
         -7.4741e-02, -4.7634e-01, -1.9773e-01,  2.4824e-01, -1.2162e-01,
          1.6678e-01,  2.1045e-01, -1.4576e-01,  1.2636e-01,  1.8635e-02,
          2.4640e-01,  5.7090e-01, -4.7014e-01,  1.3782e-01,  7.3650e-01,
         -3.3808e-01, -5.0330e-02, -1.6452e-01, -4.3517e-01, -1.2900e-01,
          1.6516e-01,  3.4004e-01, -1.4930e-01,  2.2421e-02, -1.0488e-01,
         -5.1916e-01,  3.2964e-01, -2.2162e-01, -3.4206e-01,  1.1993e-01,
         -7.0148e-01, -2.3126e-01,  1.1224e-01,  1.2550e-01, -2.5191e-01,
         -4.6374e-01, -2.7261e-02, -2.8415e-01, -9.9250e-02, -3.7018e-02,
         -8.9192e-01,  2.5005e-01,  1.5816e-01,  2.2701e-01, -2.8497e-01,
          4.5300e-01,  5.0935e-03, -7.9441e-01, -3.1007e-01, -1.7403e-01,
          4.3029e-01,  1.6816e-01,  1.0590e-01, -4.8987e-01,  3.1856e-01,
          3.